# NFL Upset Prediction: Data Exploration

This notebook explores the NFL game data and betting data to understand:
1. Data quality and coverage
2. Upset rates by season and spread
3. Team distributions and trends
4. Data gaps and issues

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Project imports
import sys
sys.path.insert(0, '..')

from src.data.nfl_loader import load_schedules
from src.data.betting_loader import load_betting_data
from src.data.merger import merge_nfl_betting_data
from src.data.verify_data import verify_data_coverage

# Display settings
pd.set_option('display.max_columns', 50)
sns.set_style('whitegrid')
%matplotlib inline

## 1. Load Data Sources

In [ ]:
# Load NFL schedules (2015-2023)
seasons = list(range(2015, 2024))
nfl_data = load_schedules(seasons=seasons)
print(f"NFL schedule records: {len(nfl_data):,}")
print(f"Seasons: {nfl_data['season'].min()} - {nfl_data['season'].max()}")
nfl_data.head()

In [ ]:
# Load betting data
betting_data = load_betting_data(min_season=2015, max_season=2023)
print(f"Betting records: {len(betting_data):,}")
print(f"Seasons: {betting_data['schedule_season'].min()} - {betting_data['schedule_season'].max()}")
betting_data.head()

In [ ]:
# Merge datasets
merged_data, audit = merge_nfl_betting_data(nfl_data, betting_data)
print(f"Merged records: {len(merged_data):,}")
print(f"Merge rate: {audit.merge_rate:.1%}")
print(f"Unmatched NFL: {len(audit.unmatched_nfl_rows):,}")
print(f"Unmatched betting: {len(audit.unmatched_betting_rows):,}")

## 2. Data Quality Check

In [ ]:
# Check missing values in key columns
key_cols = ['home_team', 'away_team', 'home_score', 'away_score', 
            'spread_favorite', 'team_favorite_id', 'over_under_line']

missing = merged_data[key_cols].isnull().sum()
print("Missing values in key columns:")
print(missing[missing > 0])

In [ ]:
# Verify data coverage by season
coverage = verify_data_coverage(merged_data)
coverage_df = pd.DataFrame(coverage['seasons']).T
coverage_df.index.name = 'season'
coverage_df

## 3. Upset Analysis

In [ ]:
# Define upset: underdog (spread >= 3) wins outright
def identify_upset(row):
    if pd.isna(row['spread_favorite']) or pd.isna(row['team_favorite_id']):
        return np.nan
    
    spread = abs(row['spread_favorite'])
    if spread < 3.0:  # Minimum spread threshold
        return np.nan
    
    # Determine underdog
    favorite = row['team_favorite_id']
    if favorite == row['home_team']:
        underdog = row['away_team']
        underdog_score = row['away_score']
        favorite_score = row['home_score']
    else:
        underdog = row['home_team']
        underdog_score = row['home_score']
        favorite_score = row['away_score']
    
    # Check if underdog won outright
    return 1 if underdog_score > favorite_score else 0

merged_data['is_upset'] = merged_data.apply(identify_upset, axis=1)

In [ ]:
# Overall upset rate
valid_games = merged_data.dropna(subset=['is_upset'])
print(f"Games with valid upset data: {len(valid_games):,}")
print(f"Overall upset rate: {valid_games['is_upset'].mean():.1%}")

In [ ]:
# Upset rate by season
upset_by_season = valid_games.groupby('season')['is_upset'].agg(['mean', 'count'])
upset_by_season.columns = ['upset_rate', 'games']

fig, ax = plt.subplots(figsize=(12, 5))
ax.bar(upset_by_season.index, upset_by_season['upset_rate'], color='steelblue', alpha=0.7)
ax.axhline(y=valid_games['is_upset'].mean(), color='red', linestyle='--', label='Overall avg')
ax.set_xlabel('Season')
ax.set_ylabel('Upset Rate')
ax.set_title('NFL Upset Rate by Season (Spread >= 3)')
ax.legend()
plt.tight_layout()
plt.savefig('../results/figures/upset_rate_by_season.png', dpi=150)
plt.show()

In [ ]:
# Add spread magnitude for analysis
valid_games = valid_games.copy()
valid_games['spread_magnitude'] = valid_games['spread_favorite'].abs()

# Upset rate by spread bucket
bins = [3, 5, 7, 10, 14, 20]
labels = ['3-5', '5-7', '7-10', '10-14', '14+']
valid_games['spread_bucket'] = pd.cut(valid_games['spread_magnitude'], bins=bins, labels=labels, right=False)

upset_by_spread = valid_games.groupby('spread_bucket')['is_upset'].agg(['mean', 'count'])
upset_by_spread.columns = ['upset_rate', 'games']

fig, ax = plt.subplots(figsize=(10, 5))
bars = ax.bar(range(len(upset_by_spread)), upset_by_spread['upset_rate'], color='coral', alpha=0.7)
ax.set_xticks(range(len(upset_by_spread)))
ax.set_xticklabels(upset_by_spread.index)
ax.set_xlabel('Spread Magnitude')
ax.set_ylabel('Upset Rate')
ax.set_title('Upset Rate by Spread Magnitude')

# Add count labels
for i, (bar, count) in enumerate(zip(bars, upset_by_spread['games'])):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01, 
            f'n={count}', ha='center', fontsize=9)

plt.tight_layout()
plt.savefig('../results/figures/upset_rate_by_spread.png', dpi=150)
plt.show()

upset_by_spread

## 4. Team Analysis

In [ ]:
# Teams with highest upset rate as underdogs
def get_underdog_team(row):
    if pd.isna(row['team_favorite_id']):
        return None
    if row['team_favorite_id'] == row['home_team']:
        return row['away_team']
    return row['home_team']

valid_games['underdog_team'] = valid_games.apply(get_underdog_team, axis=1)

underdog_performance = valid_games.groupby('underdog_team')['is_upset'].agg(['mean', 'sum', 'count'])
underdog_performance.columns = ['upset_rate', 'upsets', 'games_as_underdog']
underdog_performance = underdog_performance[underdog_performance['games_as_underdog'] >= 20]
underdog_performance = underdog_performance.sort_values('upset_rate', ascending=False)

print("Top 10 teams by upset rate when underdog (min 20 games):")
underdog_performance.head(10)

In [ ]:
# Distribution of spreads
fig, ax = plt.subplots(figsize=(12, 5))
ax.hist(valid_games['spread_magnitude'], bins=30, edgecolor='black', alpha=0.7)
ax.axvline(x=valid_games['spread_magnitude'].median(), color='red', linestyle='--', 
           label=f'Median: {valid_games["spread_magnitude"].median():.1f}')
ax.set_xlabel('Spread Magnitude (absolute value)')
ax.set_ylabel('Count')
ax.set_title('Distribution of Point Spreads')
ax.legend()
plt.tight_layout()
plt.savefig('../results/figures/spread_distribution.png', dpi=150)
plt.show()

## 5. Data Gaps Summary

In [ ]:
# Summary of data coverage issues
print("=" * 50)
print("DATA COVERAGE SUMMARY")
print("=" * 50)
print(f"\nTotal games loaded: {len(merged_data):,}")
print(f"Games with valid spread (>= 3): {len(valid_games):,}")
print(f"Games excluded (spread < 3 or missing): {len(merged_data) - len(valid_games):,}")
print(f"\nSeason range: {merged_data['season'].min()} - {merged_data['season'].max()}")
print(f"Average games per season: {len(merged_data) / len(merged_data['season'].unique()):.0f}")
print(f"\nOverall upset rate: {valid_games['is_upset'].mean():.1%}")
print(f"Class imbalance ratio: {(1-valid_games['is_upset'].mean()) / valid_games['is_upset'].mean():.1f}:1")

In [ ]:
# Save processed data for next steps
valid_games.to_parquet('../data/processed/games_with_upset_target.parquet', index=False)
print(f"Saved {len(valid_games):,} games to data/processed/games_with_upset_target.parquet")